# Exp 11: ResNet-18 5-fold

Card: [exp11_resnet18_5fold.md](../../docs/neuroslope/experiments/exp11_resnet18_5fold.md)


In [ ]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd

from training.config import ROOT
from training.config import VIP
from training.cv import CVResult
from training.runs import read_config
from training.tracking import start_run
from analysis.plots import pr_curves, save_figures, use_style
from analysis.thresholds import best_f1, cross_fold_f1
from training.config import CLASSES

CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.yaml"
EXP09 = CONFIG_DIR / "exp09_full_ratio_weight" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp11_resnet18_5fold"

# dataviz reference palette, light mode; ResNet-18 keeps its color from exp10
COLOR = "#2a78d6"
use_style()

## Training


In [ ]:
result = CVResult.load(CONFIG)
oof = result.oof

## Per-fold scores


In [ ]:
scores = result.fold_scores()
summary = pd.concat([scores, scores.agg(["mean", "std"])])
summary.round(3)

## Run-to-run noise


In [ ]:
# same setup, same folds, trained twice
noise = (scores - CVResult.load(EXP09).fold_scores()).dropna().T.add_prefix("fold ")
noise.round(3)

## Macro F1 on all out-of-fold rows


In [ ]:
same = best_f1(result.labels, result.probs, CLASSES)

cross = cross_fold_f1(result.labels, result.probs, result.oof["fold"], CLASSES)  # each fold scored with thresholds tuned on rest

pooled = pd.concat(
    {
        "tuned on same rows": same[["threshold", "f1"]],
        "tuned on other folds": cross.to_frame(),
    },
    axis=1,
)
pooled.loc["Macro F1"] = pooled.mean()
pooled.loc["Macro F1", ("tuned on same rows", "threshold")] = float("nan")
pooled.round(3)

## Precision-recall curves


In [ ]:
fig = pr_curves({"ResNet-18": result}, VIP, colors={"ResNet-18": COLOR}, title="ResNet-18, out-of-fold rows of folds 0–4")
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "full"}):
    mlflow.log_table(summary.reset_index(names="fold"), "fold_scores.json")
    mlflow.log_table(noise.reset_index(names="metric"), "noise.json")
    flat = pooled.copy()
    flat.columns = [" ".join(c) for c in flat.columns]
    mlflow.log_table(flat.reset_index(names="class"), "pooled_f1.json")

save_figures(figures, FIGURES)